# 04 — DuckDB SQL over the Catalog

This notebook demonstrates power-user SQL analysis using `labdata.duck`.

## What `labdata.duck.connect()` gives you

- A DuckDB connection with the catalog database ATTACHed read-only as schema `catalog`.
- SQL like `SELECT * FROM catalog.v_runs` works directly.
- DuckDB built-ins `read_csv_auto(...)` and `read_parquet(...)` let you query
  processed/raw CSV or Parquet files by path — without loading them into Postgres.

**Access model:** read-only catalog views and file system access via `LABDATA_ROOT`.
No writes to Postgres are possible through this connection.

In [ ]:
import pandas as pd

import labdata.duck as duck
import labdata.catalog as catalog
import labdata.store as store

# Open a DuckDB connection with catalog ATTACHed
con = duck.connect()
print("DuckDB connection opened with catalog ATTACHed read-only as schema 'catalog'")

## 1. Aggregate: measurement type counts

Count runs grouped by measurement type directly from the catalog view.

In [ ]:
result = con.sql("""
    SELECT
        measurement_type,
        count(*) AS run_count
    FROM catalog.v_runs
    GROUP BY 1
    ORDER BY run_count DESC
""")
print("Run counts by measurement type:")
result.show()

## 2. State distribution across all runs

In [ ]:
result = con.sql("""
    SELECT
        state,
        count(*) AS count,
        round(100.0 * count(*) / sum(count(*)) OVER (), 1) AS pct
    FROM catalog.v_runs
    GROUP BY state
    ORDER BY count DESC
""")
print("Run state distribution:")
result.show()

## 3. JOIN: runs with their published results

Join `catalog.v_runs` with `catalog.v_published_results` to see which runs have
published outputs and who published them.

In [ ]:
result = con.sql("""
    SELECT
        r.id          AS run_id,
        r.sample_id,
        r.device_id,
        r.measurement_type,
        r.state,
        pr.id         AS published_result_id,
        pr.parser_version,
        pr.published_by,
        pr.published_at
    FROM catalog.v_runs r
    JOIN catalog.v_published_results pr
        ON pr.run_id = r.id
    ORDER BY pr.published_at DESC
    LIMIT 20
""")
print("Runs with published results (most recent first):")
result.show()

## 4. read_csv_auto over a raw or processed CSV file

DuckDB can read CSV files directly from disk using `read_csv_auto(path)`.
Here we find a published run, resolve its raw data path via `labdata.store`,
and query the CSV directly from DuckDB — without loading it into Python memory first.

This is especially useful for large files or batch analysis across many runs.

In [ ]:
import labdata.catalog as catalog

df_pub = catalog.list_runs(state='published')

if df_pub.empty:
    print("No published runs — skipping file read example.")
else:
    run_id = df_pub.iloc[0]['id']
    raw_files = store.raw_files(run_id)

    # Find the first CSV-like file
    csv_files = [p for p in raw_files if p.suffix in ('.data', '.csv', '.dat', '.txt')]

    if not csv_files:
        print(f"No CSV-like raw file found for run {run_id}.")
    else:
        csv_path = str(csv_files[0])
        print(f"Reading: {csv_path}")

        # DuckDB reads the CSV file; skip comment lines starting with '#'
        result = con.sql(f"""
            SELECT *
            FROM read_csv_auto('{csv_path}', comment='#', header=true)
            LIMIT 10
        """)
        print("First 10 rows from raw CSV:")
        result.show()

## 5. read_csv_auto with statistics

Use DuckDB aggregation directly on the file for quick statistics without
loading the full dataset into Python.

In [ ]:
if not df_pub.empty and csv_files:
    csv_path = str(csv_files[0])

    result = con.sql(f"""
        SELECT
            count(*) AS rows,
            min(COLUMNS(*))  AS col_min,
            max(COLUMNS(*))  AS col_max,
            avg(COLUMNS(*))  AS col_avg
        FROM read_csv_auto('{csv_path}', comment='#', header=true)
    """)
    print("CSV statistics:")
    result.show()

In [ ]:
# Close connection when done
con.close()
print("DuckDB connection closed.")